# **Initialization**

In [2]:
print('Start')

Start


In [3]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import glob
import pulp
import vrplib
import re
import os
import gc
import contextlib
import modified_didppy as m_dp
import time as pytime

# **Data**

In [4]:
# Directory containing the VRP instances (Update this path)
# Note: Ensure this folder contains your .vrp files (e.g., A-n33-k5.vrp)
folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\X"
#folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\A"
# Get all .vrp files
all_files = glob.glob(os.path.join(folder_path, "*.vrp"))

# Select random instances (or all)
num_instances_to_test = 1000
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("Selected Instances:")
for f in selected_files:
    print(f" - {os.path.basename(f)}")
print("-" * 50)

# ==========================================
# 2. Data Reading & Model Definition
# ==========================================

# Global variables to store current instance data (used by model creator)
current_num_locations = 0
current_num_vehicles = 0
current_capacity = 0
current_cust_demands = []
current_travel_cost = []

def read_formatted_data(file_path):
    """
    Reads a VRP file using vrplib and updates the global variables.
    """
    global current_num_locations, current_num_vehicles, current_capacity
    global current_cust_demands, current_travel_cost
    
    # Read instance
    instance = vrplib.read_instance(file_path)
    
    # Extract Capacity & Dimensions
    #current_capacity = instance['capacity']
    #current_num_locations = instance['dimension']
    
    # Extract Demands (Includes depot at index 0 with demand 0)
    #current_cust_demands = instance['demand'].tolist()
    
    # Extract/Compute Edge Weights
    # vrplib usually computes euclidean distance automatically in 'edge_weight'
    #current_travel_cost = instance['edge_weight'].tolist()
    
    # Extract Capacity & Dimensions
    current_capacity = float(instance['capacity'])
    current_num_locations = int(instance['dimension'])

    # Extract Demands (Includes depot at index 0 with demand 0)
    current_cust_demands = [float(x) for x in instance['demand']]

    # Extract/Compute Edge Weights
    # vrplib usually computes euclidean distance automatically in 'edge_weight'
    current_travel_cost = [[float(x) for x in row] for row in instance['edge_weight']]
    
    # Try to determine number of vehicles from comment or filename
    # Defaulting to a safe upper bound (e.g., N) or a specific number if known
    # For Augerat instances (A-n33-k5), 'k5' means 5 trucks.
    import re
    match_comment = re.search(r"No of trucks:\s*(\d+)", str(instance.get('comment', '')))
    match_name = re.search(r"-k(\d+)", os.path.basename(file_path))
    
    if match_comment:
        current_num_vehicles = int(match_comment.group(1))
    elif match_name:
        current_num_vehicles = int(match_name.group(1))
    else:
        # Fallback: Estimate or set a high number (e.g., N)
        current_num_vehicles = current_num_locations 

    return current_num_locations, current_num_vehicles, current_travel_cost, current_capacity, current_cust_demands

def get_best_known_solution(vrp_file_path):
    """
    Looks for a corresponding .sol file in the same directory 
    and returns the 'cost' from it using vrplib.
    """
    # Construct expected solution path (e.g., replace .vrp with .sol)
    sol_path = vrp_file_path.rsplit('.', 1)[0] + '.sol'
    
    if os.path.exists(sol_path):
        try:
            solution = vrplib.read_solution(sol_path)
            # Return cost, defaulting to None if key missing
            return solution.get('cost', None) 
        except Exception as e:
            print(f"Warning: Could not read solution file {sol_path}: {e}")
            return None
    else:
        # File doesn't exist
        return None

Found 100 files. Selected 100 for testing.
Selected Instances:
 - X-n1001-k43.vrp
 - X-n101-k25.vrp
 - X-n106-k14.vrp
 - X-n110-k13.vrp
 - X-n115-k10.vrp
 - X-n120-k6.vrp
 - X-n125-k30.vrp
 - X-n129-k18.vrp
 - X-n134-k13.vrp
 - X-n139-k10.vrp
 - X-n143-k7.vrp
 - X-n148-k46.vrp
 - X-n153-k22.vrp
 - X-n157-k13.vrp
 - X-n162-k11.vrp
 - X-n167-k10.vrp
 - X-n172-k51.vrp
 - X-n176-k26.vrp
 - X-n181-k23.vrp
 - X-n186-k15.vrp
 - X-n190-k8.vrp
 - X-n195-k51.vrp
 - X-n200-k36.vrp
 - X-n204-k19.vrp
 - X-n209-k16.vrp
 - X-n214-k11.vrp
 - X-n219-k73.vrp
 - X-n223-k34.vrp
 - X-n228-k23.vrp
 - X-n233-k16.vrp
 - X-n237-k14.vrp
 - X-n242-k48.vrp
 - X-n247-k50.vrp
 - X-n251-k28.vrp
 - X-n256-k16.vrp
 - X-n261-k13.vrp
 - X-n266-k58.vrp
 - X-n270-k35.vrp
 - X-n275-k28.vrp
 - X-n280-k17.vrp
 - X-n284-k15.vrp
 - X-n289-k60.vrp
 - X-n294-k50.vrp
 - X-n298-k31.vrp
 - X-n303-k21.vrp
 - X-n308-k13.vrp
 - X-n313-k71.vrp
 - X-n317-k53.vrp
 - X-n322-k28.vrp
 - X-n327-k20.vrp
 - X-n331-k15.vrp
 - X-n336-k84.vrp
 - 

# **DIDP model**

In [5]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    # =========================================================
    # 1. Define Data
    # =========================================================
    n = current_num_locations
    m = current_num_vehicles
    q = current_capacity
    # Weights (demand)
    d = current_cust_demands

    # Distance matrix
    distance_list = current_travel_cost
    
    # =========================================================
    # 2. Define DIDP model
    # =========================================================
    model = m_dp.Model(float_cost= True)

    customer = model.add_object_type(number=n)
    unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name='unvisited_customers')
    location_var = model.add_element_var(object_type=customer, target=0)
    load_var = model.add_float_resource_var(target=0, less_is_better=True)
    vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)

    weight = model.add_float_table(d)
    distance_table = model.add_float_table(distance_list)

    model.add_base_case([unvisited_var.is_empty(), location_var == 0])

    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}",
            cost=distance_table[location_var, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, load_var + weight[j]),
            ],
            preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
        )
        model.add_transition(visit)

    for j in range(1, n):
        visit_via_depot = m_dp.Transition(
            name=f"visit {j} with new vehicle",
            cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, weight[j]),
                (vehicles_var, vehicles_var + 1),
            ],
            preconditions=[unvisited_var.contains(j), vehicles_var < m],
        )
        model.add_transition(visit_via_depot)

    return_to_depot = m_dp.Transition(
        name="return",
        cost=distance_table[location_var, 0] + m_dp.FloatExpr.state_cost(),
        effects=[(location_var, 0)],
        preconditions=[unvisited_var.is_empty(), location_var != 0],
    )
    model.add_transition(return_to_depot)

    model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])

    # Min outgoing edge
    min_to = model.add_float_table(
        [min(distance_list[k][j] for k in range(n) if k != j) for j in range(n)]
    )
    model.add_dual_bound(min_to[unvisited_var] + (location_var != 0).if_then_else(min_to[0], 0))

    # Min incoming edge
    min_from = model.add_float_table(
        [min(distance_list[j][k] for k in range(n) if k != j) for j in range(n)]
    )
    model.add_dual_bound(
        min_from[unvisited_var] + (location_var != 0).if_then_else(min_from[location_var], 0)
    )
    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unvisited_var": unvisited_var,
        "location_var": location_var,
        "distance_matrix": distance_list,
        "demand": d,
        "capacity": q,
        "num_vehicles": m,
        "num_nodes": n
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

# **Execution**

In [11]:
results_data = []
output_csv_name = "CVRP_single_dual_bound_A_set_results_10s_lim.csv"

for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    
    # --- NEW: Get Best Known Solution ---
    best_known_cost = get_best_known_solution(file_path)
    
    try:
        # --- A. Read Data & Update Globals ---
        current_num_locations, current_num_vehicles, current_travel_cost, current_capacity, current_cust_demands = read_formatted_data(file_path)
        
        # --- B. Initialize Model ---
        model, state_data = creation_of_didp_model_function()
        
        # --- C. Solver Execution ---
        t_start = pytime.time()
        
        # Standard CABS solver with limit
        solver = m_dp.CABS(
            model,
            quiet=False,
            time_limit=10 # 30 seconds limit as per your previous code
        )
        
        solution = solver.search()
        
        t_end = pytime.time()
        duration = t_end - t_start
        
        # --- D. Logging Results ---
        if solution.is_optimal:
            cost = solution.cost
            status = "True"
        elif solution.cost is not None:
            cost = solution.cost
            status = "False (Time Limit)"
        else:
            cost = float('inf') # Use float inf for better sorting later
            status = "False (No Sol)"

        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        
        # Calculate Gap if best known exists and we have a solution
        gap = "N/A"
        if best_known_cost is not None and cost != float('inf') and cost != "Inf":
            try:
                gap = round(((cost - best_known_cost) / best_known_cost) * 100, 2)
                gap = f"{gap}%"
            except:
                gap = "Error"

        print(f"   -> My Cost: {cost} | Best Known: {best_known_cost} | Gap: {gap}")
        print(f"   -> Time: {duration:.2f}s | Optimal: {status}")

        results_data.append({
            "Instance": instance_name,
            "Best Known cost": best_known_cost, # <--- NEW COLUMN
            "Cost": cost,
            "Gap to BKS": gap,                      # <--- Optional Helper Column
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": status,
            "Infeasibility": solution.is_infeasible
        })

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        import traceback
        traceback.print_exc()
        
        results_data.append({
            "Instance": instance_name,
            "Best Known cost": best_known_cost,
            "Cost": "Error",
            "Gap to BKS": "N/A",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error"
        })

    # --- E. Intermediate Save ---
    df_results = pd.DataFrame(results_data)
    df_results.to_csv(output_csv_name, index=False)

print("\n" + "="*50)
print("Batch Testing Complete.")
print(f"Results saved to {output_csv_name}")
print(df_results[['Instance', 'Best Known cost', 'Cost', 'Gap to BKS', 'Running Time (s)']])


[1/27] Processing: A-n32-k5.vrp
   -> My Cost: 843.4253891634958 | Best Known: 784 | Gap: 7.58%
   -> Time: 10.25s | Optimal: False (Time Limit)

[2/27] Processing: A-n33-k5.vrp
   -> My Cost: 707.5806014626186 | Best Known: 661 | Gap: 7.05%
   -> Time: 10.02s | Optimal: False (Time Limit)

[3/27] Processing: A-n33-k6.vrp
   -> My Cost: 825.207946480216 | Best Known: 742 | Gap: 11.21%
   -> Time: 10.02s | Optimal: False (Time Limit)

[4/27] Processing: A-n34-k5.vrp
   -> My Cost: 920.1110817024095 | Best Known: 778 | Gap: 18.27%
   -> Time: 10.02s | Optimal: False (Time Limit)

[5/27] Processing: A-n36-k5.vrp
   -> My Cost: 919.6978950436514 | Best Known: 799 | Gap: 15.11%
   -> Time: 10.01s | Optimal: False (Time Limit)

[6/27] Processing: A-n37-k5.vrp
   -> My Cost: 748.5032643124983 | Best Known: 669 | Gap: 11.88%
   -> Time: 10.02s | Optimal: False (Time Limit)

[7/27] Processing: A-n37-k6.vrp
   -> My Cost: 1006.3905549707479 | Best Known: 949 | Gap: 6.05%
   -> Time: 10.01s | Op

# **Testing 1 instance**

In [7]:
alone_test_instance =[
    'C:\\Users\\ACER\\Desktop\\Code\\0_Thesis_implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n1001-k43.vrp'
]
for i, file_path in enumerate(alone_test_instance):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(alone_test_instance)}] Processing: {instance_name}")
    # --- A. Read Data & Update Globals ---
    current_num_locations, current_num_vehicles, current_travel_cost, current_capacity, current_cust_demands = read_formatted_data(file_path)
    #current_capacity = current_capacity + 1000
    # --- B. Initialize Model ---
    model, state_data = creation_of_didp_model_function()
    # --- C. Solver Execution ---
    t_start = pytime.time()
    # Standard CABS solver with 30-minute limit
    solver = m_dp.CABS(
        model,
        quiet=False,
        time_limit= 1200
    )
    solution = solver.search()

    t_end = pytime.time()
    duration = t_end - t_start

    # --- D. Logging Results ---
    if solution.is_optimal:
        cost = solution.cost
        status = "True"
    elif solution.cost is not None:
        cost = solution.cost
        status = "False (Time Limit)"
    else:
        cost = "Inf"
        status = "False (No Sol)"

    nodes_gen = solution.generated
    nodes_exp = solution.expanded
    print(f"Number of lcations: {current_num_locations}")
    print(f"Number of vehicles: {current_num_vehicles}")
    print(f"Total Capacity: {current_capacity*current_num_vehicles}")
    print(f"Total Demand: {sum(current_cust_demands)}")
    print(f"   -> Done. Cost: {cost}, Time: {duration:.2f}s, Optimal: {status}")
    print(f"      Nodes Expanded: {nodes_exp}, Nodes Generated: {nodes_gen}")
    print("---Transition sequence--")
    for t in solution.transitions:
        print(f"   - {t.name}")


[1/1] Processing: X-n1001-k43.vrp
Number of lcations: 1001
Number of vehicles: 43
Total Capacity: 5633.0
Total Demand: 5557.0
   -> Done. Cost: 86711.113622368, Time: 1200.23s, Optimal: False (Time Limit)
      Nodes Expanded: 16951, Nodes Generated: 123368
---Transition sequence--
   - visit 658
   - visit 408
   - visit 698
   - visit 664
   - visit 768
   - visit 662
   - visit 335
   - visit 873
   - visit 18
   - visit 930
   - visit 350
   - visit 601
   - visit 260
   - visit 12
   - visit 908
   - visit 13
   - visit 522
   - visit 742
   - visit 55
   - visit 230
   - visit 507
   - visit 537
   - visit 115
   - visit 179
   - visit 483
   - visit 819 with new vehicle
   - visit 801
   - visit 748
   - visit 876
   - visit 515
   - visit 958
   - visit 766
   - visit 736
   - visit 457
   - visit 903
   - visit 187
   - visit 237
   - visit 816
   - visit 822
   - visit 921
   - visit 524
   - visit 900
   - visit 935
   - visit 151
   - visit 735
   - visit 320
   - visit 23

# **Selecting instances**

In [24]:
#Manually select specific instances for testing
set_A_selected_instances = [
    "A-n39-k6.vrp",
    "A-n48-k7.vrp",
    "A-n55-k9.vrp",
    "A-n69-k9.vrp",
    "A-n80-k10.vrp",
]

# Build the full path for each selected file
set_A_selected_files = [os.path.join(folder_path, fname) for fname in set_A_selected_instances]

print(f"Selected {len(set_A_selected_files)} files for partitioning.")

# Read the full results CSV
df = pd.read_csv("CVRP_single_dual_bound_A_set_results_10s_lim.csv")

# Filter for selected instances only
df_selected = df[df["Instance"].isin(set_A_selected_instances)]

# Save to a new CSV
df_selected.to_csv("CVRP_selected_A_set_results.csv", index=False)

Selected 5 files for partitioning.


In [27]:
#Manually select specific instances for testing
set_X_selected_instances = [
    "X-n106-k14.vrp",
    "X-n162-k11.vrp",
    "X-n181-k23.vrp",
    "X-n204-k19.vrp",
    "X-n190-k8.vrp",
    #"X-n209-k16.vrp" work but slow
]

# Build the full path for each selected file
set_X_selected_files = [os.path.join(folder_path, fname) for fname in set_X_selected_instances]

print(f"Selected {len(set_X_selected_files)} files for partitioning.")

# Read the full results CSV
df = pd.read_csv("CVRP_single_dual_bound_X_set_results_10s_lim.csv")

# Filter for selected instances only
df_selected = df[df["Instance"].isin(set_X_selected_instances)]

# Save to a new CSV
df_selected.to_csv("CVRP_selected_X_set_results.csv", index=False)

Selected 5 files for partitioning.


In [ ]:
# Combine A and X selected results into one CSV

# Read the selected A and X results
df_A = pd.read_csv("CVRP_selected_A_set_results.csv")
df_X = pd.read_csv("CVRP_selected_X_set_results.csv")

# Combine them
df_combined = pd.concat([df_A, df_X], ignore_index=True)

# Save to the new CSV
df_combined.to_csv("CVRP_single_dual_bound_selected_results_10s_lim.csv", index=False)

print("Combined CSV created: CVRP_single_dual_bound_results_10s_lim.csv")
print(f"Total rows: {len(df_combined)}")

Combined CSV created: CVRP_single_dual_bound_results_10s_lim.csv
Total rows: 10


# **Run single dual bound model on selected instance**

In [7]:
import os
import csv
import time as pytime
import modified_didppy as m_dp
import gc

# =========================================================
# 1. BATCH CONFIGURATION
# =========================================================
# Directories containing your .vrp files
SEARCH_DIRS = [
    r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\A",
    r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\X",
]

# The specific instances you selected
SELECTED_INSTANCES = [
    # Set A
    "A-n39-k6.vrp",
    "A-n48-k7.vrp",
    "A-n55-k9.vrp",
    "A-n69-k9.vrp",
    "A-n80-k10.vrp",
    # Set X
    "X-n106-k14.vrp",
    "X-n162-k11.vrp",
    "X-n181-k23.vrp",
    "X-n204-k19.vrp",
    "X-n190-k8.vrp",
]

OUTPUT_CSV = "CVRP_single_dual_bound_selected_results_1800s_lim.csv"
SOLVER_TIME_LIMIT = 1800  # 30 Minutes

# =========================================================
# 2. BATCH HELPER UTILITIES
# =========================================================
def find_file_in_dirs(instance_name, search_dirs):
    """Helper to find the full path of a .vrp file."""
    for folder in search_dirs:
        path = os.path.join(folder, instance_name)
        if os.path.exists(path): return path
    return None

def get_processed_instances(csv_path):
    """Returns a set of instance names already present in the CSV."""
    if not os.path.exists(csv_path):
        return set()
    try:
        df = pd.read_csv(csv_path)
        return set(df['Instance'].unique())
    except:
        return set()

# =========================================================
# 3. MAIN EXECUTION LOOP
# =========================================================

# A. Resume Check
processed = get_processed_instances(OUTPUT_CSV)
print(f"🚀 Starting Batch Run (Limit: {SOLVER_TIME_LIMIT}s)")
print(f"   -> Found {len(processed)} already completed instances in {OUTPUT_CSV}.")

for i, instance_name in enumerate(SELECTED_INSTANCES):
    # 1. Skip if done
    if instance_name in processed:
        continue
        
    print(f"\n[{i+1}/{len(SELECTED_INSTANCES)}] Processing: {instance_name}")
    
    # 2. Find File
    file_path = find_file_in_dirs(instance_name, SEARCH_DIRS)
    if not file_path:
        print(f"   ⚠️ File not found in search directories. Skipping.")
        continue
        
    # 3. Get BKS (Using your existing function)
    bks = get_best_known_solution(file_path)
    
    try:
        # 4. Update Globals & Create Model
        # Call YOUR existing function (updates globals: current_num_locations, etc.)
        read_formatted_data(file_path)
        
        # Create DIDP model using the updated globals
        model, _ = creation_of_didp_model_function()
        
        # 5. Run Solver
        t_start = pytime.time()
        solver = m_dp.CABS(model, quiet=False, time_limit=SOLVER_TIME_LIMIT)
        solution = solver.search()
        duration = pytime.time() - t_start
        
        # 6. Parse Results
        cost = solution.cost if solution.cost is not None else float('inf')
        is_optimal = solution.is_optimal
        status_str = "Optimal" if is_optimal else ("Time Limit" if cost != float('inf') else "No Sol")
        
        # Calculate Gap
        gap_str = "N/A"
        if bks and cost != float('inf'):
            try:
                gap_val = round(((cost - bks) / bks) * 100, 2)
                gap_str = f"{gap_val}%"
            except: gap_str = "Error"

        print(f"   -> Cost: {cost} | BKS: {bks} | Gap: {gap_str}")
        print(f"   -> Time: {duration:.2f}s | Status: {status_str}")

        # 7. Save to CSV
        result_row = {
            "Instance": instance_name,
            "Best Known Cost": bks,
            "Cost": cost,
            "Gap to BKS": gap_str,
            "Nodes Expanded": solution.expanded,
            "Nodes Generated": solution.generated,
            "Running Time (s)": duration,
            "Is Optimal": status_str,
            "Infeasibility": solution.is_infeasible
        }
        
        file_exists = os.path.exists(OUTPUT_CSV)
        with open(OUTPUT_CSV, mode='a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=result_row.keys())
            if not file_exists: writer.writeheader()
            writer.writerow(result_row)
            
        processed.add(instance_name)
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        import traceback
        traceback.print_exc()

    finally:
        gc.collect()

print("\n🎉 Selected Instances Batch Complete!")

🚀 Starting Batch Run (Limit: 1800s)
   -> Found 0 already completed instances in CVRP_single_dual_bound_selected_results_1800s_lim.csv.

[1/10] Processing: A-n39-k6.vrp
   -> Cost: 877.8464680622174 | BKS: 831 | Gap: 5.64%
   -> Time: 1801.40s | Status: Time Limit

[2/10] Processing: A-n48-k7.vrp
   -> Cost: 1150.2588516046962 | BKS: 1073 | Gap: 7.2%
   -> Time: 1800.43s | Status: Time Limit

[3/10] Processing: A-n55-k9.vrp
   -> Cost: 1161.8729164888778 | BKS: 1073 | Gap: 8.28%
   -> Time: 1800.56s | Status: Time Limit

[4/10] Processing: A-n69-k9.vrp
   -> Cost: 1289.682883345472 | BKS: 1159 | Gap: 11.28%
   -> Time: 1800.31s | Status: Time Limit

[5/10] Processing: A-n80-k10.vrp
   -> Cost: 2071.8941783684795 | BKS: 1763 | Gap: 17.52%
   -> Time: 1800.13s | Status: Time Limit

[6/10] Processing: X-n106-k14.vrp
   -> Cost: 27865.19902137087 | BKS: 26362 | Gap: 5.7%
   -> Time: 1800.13s | Status: Time Limit

[7/10] Processing: X-n162-k11.vrp
   -> Cost: 15768.499316267662 | BKS: 14138